# 🎬 H3 Studio — Colab Cloud Inference & Resilient Chained Pipeline
### MiniMax-H3 Multimodal Storyboard Studio with Anti-503 Checkpoint Recovery

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/h3-studio/h3-studio/blob/main/notebooks/H3_Studio_Colab.ipynb)

> **Key Architectural Highlights**:
> 1. **Zero Path Nesting**: Guarantees clean `/content/H3-Studio` workspace without phantom `/content/content/` bugs.
> 2. **FUSE I/O Bypass**: Fast sequential tar extraction from Google Drive to local NVMe SSD (`/content/models`), avoiding random seek latency on 30GB+ weights.
> 3. **Anti-503 Checkpoint State Machine**: Every 5-second segment is verified via FFmpeg (0 black/frozen frames) and committed to Google Drive in real time. If the Colab session disconnects, re-running resumes from the exact unrendered segment.

## Step 1: GPU Inspection & Hardware Acceleration

In [ ]:
!nvidia-smi

import torch
print(f"CUDA Available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device     : {torch.cuda.get_device_name(0)}")
    print(f"Device Memory  : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GiB")

## Step 2: Clone Repository & Normalize Workspace (No Path Nesting)

In [ ]:
import os, sys
from pathlib import Path

# Explicitly anchor to /content/H3-Studio to prevent /content/content/ nested directories
WORKSPACE_ROOT = Path("/content/H3-Studio").resolve()
if not (WORKSPACE_ROOT / ".git").is_dir():
    !git clone https://github.com/h3-studio/h3-studio.git {WORKSPACE_ROOT}

%cd {WORKSPACE_ROOT}
if str(WORKSPACE_ROOT) not in sys.path:
    sys.path.insert(0, str(WORKSPACE_ROOT))

print(f"✅ Workspace configured at: {WORKSPACE_ROOT}")
!pip install -q -r requirements.txt
!pip install -q safetensors huggingface_hub

## Step 3: Google Drive Mount & High-Speed NVMe Weight Staging

> 💡 **Why we copy to `/content/models` (NVMe SSD)**:
> Google Drive's network filesystem (FUSE) has severe I/O latency when reading 30GB+ `.safetensors` model weights with random seeks. Copying or unpacking from a Drive `.tar` package to local NVMe is up to **10x faster** than direct FUSE reading.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/H3-Studio')
DRIVE_MODELS = DRIVE_ROOT / 'models'
DRIVE_OUTPUTS = DRIVE_ROOT / 'outputs'
LOCAL_MODELS = Path('/content/models/MiniMax-H3-Turbo')

DRIVE_OUTPUTS.mkdir(parents=True, exist_ok=True)
LOCAL_MODELS.parent.mkdir(parents=True, exist_ok=True)

# Check for pre-packaged model archive in Google Drive
drive_tar = DRIVE_MODELS / 'MiniMax-H3-Turbo.tar'
if not LOCAL_MODELS.is_dir():
    if drive_tar.is_file():
        print(f"📦 Unpacking cached weights from Google Drive: {drive_tar} ...")
        !tar -xf {drive_tar} -C /content/models/
        print("✅ Local NVMe model unpack complete!")
    else:
        print("⬇️ Downloading MiniMax-H3 weights from Hugging Face Hub...")
        from huggingface_hub import snapshot_download
        snapshot_download(repo_id='MiniMaxAI/MiniMax-H3', local_dir=str(LOCAL_MODELS))
        print("✅ Download complete!")
        # Optionally create archive in Drive for future one-click launches
        DRIVE_MODELS.mkdir(parents=True, exist_ok=True)
        print(f"💾 Archiving weights to Google Drive for instant future sessions: {drive_tar} ...")
        !tar -cf {drive_tar} -C /content/models MiniMax-H3-Turbo
        print("✅ Drive model cache created!")
else:
    print(f"✅ Local model directory ready at: {LOCAL_MODELS}")

## Step 4: Choose Storyboard Template & Pre-Flight Validation

In [ ]:
# Available Golden Templates:
# 1. examples/storyboards/cyberpunk_detective_25s.json (Neo-Noir Detective 25s)
# 2. examples/storyboards/xianxia_aerial_duel_25s.json (Wuxia Aerial Duel 25s)
SELECTED_MANIFEST = "examples/storyboards/cyberpunk_detective_25s.json"

!python3 pipeline/h3_storyboard_pipeline.py --dry-run --chain-manifest {SELECTED_MANIFEST} --mode turbo

## Step 5: Execute 25s Chained Pipeline with Anti-503 Checkpoint Recovery

In [ ]:
# Execute with model override pointing to high-speed NVMe storage
!python3 pipeline/h3_storyboard_pipeline.py \
    --chain-manifest {SELECTED_MANIFEST} \
    --model-dir {LOCAL_MODELS} \
    --mode turbo \
    --export-gdrive

# Instant Google Drive sync for state and outputs
!rsync -av outputs/ /content/drive/MyDrive/H3-Studio/outputs/
print("☁️ Google Drive outputs and checkpoint states synchronized!")

## Step 6: In-Browser Video Player

In [ ]:
from IPython.display import HTML
from base64 import b64encode
import glob

merged_videos = sorted(glob.glob("outputs/**/*_merged*.mp4", recursive=True))
if merged_videos:
    latest_video = merged_videos[-1]
    print(f"🎬 Displaying final merged video: {latest_video}")
    mp4_bytes = open(latest_video, 'rb').read()
    data_url = "data:video/mp4;base64," + b64encode(mp4_bytes).decode()
    HTML(f'''
    <video width="864" height="480" controls autoplay style="border-radius:8px; box-shadow:0 4px 12px rgba(0,0,0,0.3);">
        <source src="{data_url}" type="video/mp4">
        Your browser does not support the video tag.
    </video>
    ''')
else:
    print("No merged video found yet.")